In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ONE-Lab/MixSet")

README.md:   0%|          | 0.00/4.07k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Mixset_train.json:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Mixset_test.json:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/600 [00:00<?, ? examples/s]

In [2]:
print(ds['train'][1])  # Print the second sample (index 1) from the 'train' split of the Hugging Face Dataset

{'category': 'blog', 'id': 1, 'original': "Summer tv tv in the summer used to be a dead zone full of repeats or something. But now there are two great new shows on tv and i have been watching both religiously for the last week. Blow out the gayest concept ever that seemingly has zero gay people in sight. Hairstylist jonothan antin wants build a second location to his incredibly popular west hollywood salon but this time in beverly hills. Normally no one would care, certainly not i, but as i was flipping thru the channels i caught a glipse of jonothan, equally narcissistic and dull looking at a empty space on rodeo dr speak (in a public voice) 'i am going to build here the greatest salon in the history of the world ever!' and i was hooked. Not since rocco have i found such a moron compelling tv watching.", 'revised': 'Summer TV used to be a dead zone filled with reruns, but now there are two great new shows that I\'ve been watching religiously for the past week. One show features hairst

In [3]:
# =====================================================================
# INSPECT DATASET STRUCTURE, ROW COUNT, AND COLUMN COUNT
# =====================================================================

import pandas as pd  # DataFrame library for tabular inspection

# Convert the dataset's 'train' split to a pandas DataFrame for easy structural inspection
df = ds['train'].to_pandas()  # Convert Hugging Face Dataset split into a pandas DataFrame

print("=" * 50)  # Top banner border
print("DATASET OVERVIEW:")  # Section header
print("=" * 50)  # Bottom banner border
print(f"- Rows: {df.shape[0]}")  # Print total number of rows
print(f"- Columns: {df.shape[1]}")  # Print total number of columns
print(f"- Column names: {list(df.columns)}")  # Print list of column names

print("\n" + "=" * 50)  # Top border for data types section
print("DATA TYPES:")  # Section header
print("=" * 50)  # Bottom border for data types section
print(df.dtypes)  # Print column data types

print("\n" + "=" * 50)  # Top border for preview section
print("FIRST 5 ROWS OF THE DATASET:")  # Section header
print("=" * 50)  # Bottom border for preview section
print(df.head())  # Display the first 5 rows of the DataFrame

DATASET OVERVIEW:
- Rows: 3000
- Columns: 7
- Column names: ['category', 'id', 'original', 'revised', 'mixset_category', 'binary', 'model']

DATA TYPES:
category           object
id                  int64
original           object
revised            object
mixset_category    object
binary             object
model              object
dtype: object

FIRST 5 ROWS OF THE DATASET:
  category  id                                           original  \
0     blog   0  Another week just flew by me. I drove by the n...   
1     blog   1  Summer tv tv in the summer used to be a dead z...   
2     blog   2  What a week ... My right shoulder has tendonit...   
3     blog   3  Kindness of strangers... Attempted laundry tod...   
4     blog   4  So it is 11:14pm mst, and here i sit in my bed...   

                                             revised      mixset_category  \
0  Another week has flown by. I drove past the ne...  2llama_polish_token   
1  Summer TV used to be a dead zone filled with r...

In [4]:
# Print unique values in the 'binary' column and the count of each class
print("Unique values in the 'binary' column:")  # Log unique label values
print(df['binary'].unique())  # Display distinct label values in 'binary'

print("\nLabel distribution counts:")  # Log class frequency summary
print(df['binary'].value_counts())  # Count occurrences of each unique label (including non-null values)

Unique values in the 'binary' column:
['MGT' 'HWT']

Label distribution counts:
binary
MGT    2500
HWT     500
Name: count, dtype: int64


In [5]:

# =====================================================================
# IMPROVED PIPELINE: UNDERSAMPLING + ELECTRA + VOTING + SVM TUNING
# =====================================================================

import torch  # Import PyTorch for deep learning operations
import numpy as np  # Import NumPy for numerical computations
import pandas as pd  # Import Pandas for data manipulation
import nltk  # Import NLTK for natural language processing

nltk.download('punkt', quiet=True)  # Download sentence tokenization data
nltk.download('punkt_tab', quiet=True)  # Download additional sentence tokenization data

from nltk.tokenize import sent_tokenize  # Import the sentence tokenizer
from transformers import AutoTokenizer, AutoModelForTokenClassification  # Import ELECTRA tokenizer and model
from sklearn.model_selection import train_test_split  # Import the train-test splitting function
from sklearn.svm import SVC  # Import the Support Vector Machine classifier
from sklearn.utils import resample  # Import the resampling function
from sklearn.metrics import classification_report, accuracy_score, precision_recall_curve  # Import evaluation metrics

# 1. CHECK HARDWARE DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Select GPU if available, otherwise use CPU
print(f"🚀 Using device: {device}")  # Display the selected hardware device

# =====================================================================
# STEP 1: LOAD AND PREPROCESS THE MIXSET DATASET
# =====================================================================

try:
    if 'ds' in globals():  # Check whether the dataset variable already exists
        df = ds['train'].to_pandas()  # Convert the Hugging Face dataset into a Pandas DataFrame
    else:
        df = pd.read_csv("mixset.csv")  # Load the dataset from a CSV file
    print(f"✅ MixSet dataset loaded: {len(df)} rows.")  # Display the number of loaded rows
except Exception as e:
    print(f"⚠️ Data loading error: {e}")  # Display the error if loading fails

# Convert labels: 'MGT' -> 1 (AI/Mixed), 'HWT' -> 0 (Human)
df['target'] = df['binary'].apply(lambda x: 1 if str(x).upper() == 'MGT' else 0)  # Encode text labels as binary values

# =====================================================================
# STEP 2: SPLIT TRAIN/TEST SETS BEFORE SENTENCE SEGMENTATION
# (PREVENT DATA LEAKAGE)
# =====================================================================

# Split at the original document level to prevent sentences from the same document
# from appearing in both the training and testing sets
train_df, test_df = train_test_split(
    df,  # Use the complete dataset
    test_size=0.2,  # Allocate 20% of the data to testing
    random_state=42,  # Ensure reproducible splitting
    stratify=df['target']  # Preserve the original class distribution
)

# =====================================================================
# STEP 3: APPLY UNDERSAMPLING TO BALANCE THE TRAINING SET
# =====================================================================

train_hwt = train_df[train_df['target'] == 0]  # Select Human-generated documents
train_mgt = train_df[train_df['target'] == 1]  # Select AI/Mixed-generated documents

# Reduce MGT samples to 1.5 times the number of HWT samples
# Set multiplier = 1.0 for an exact 1:1 class balance
multiplier = 1.5  # Define the desired class ratio
n_samples_mgt = min(len(train_mgt), int(len(train_hwt) * multiplier))  # Calculate the number of MGT samples to retain

train_mgt_downsampled = resample(
    train_mgt,  # Select the majority class for undersampling
    replace=False,  # Sample without replacement
    n_samples=n_samples_mgt,  # Specify the number of samples to retain
    random_state=42  # Ensure reproducible sampling
)

train_df_balanced = pd.concat([train_hwt, train_mgt_downsampled]).sample(frac=1, random_state=42)  # Combine and shuffle both classes

print("\n📊 Training set distribution AFTER undersampling:")  # Display the training set distribution
print(train_df_balanced['target'].value_counts())  # Count samples in each training class

print("\n📊 Test set distribution (original distribution):")  # Display the original test set distribution
print(test_df['target'].value_counts())  # Count samples in each test class

# Define a function to extract sentences from a DataFrame
def extract_sentences_from_df(dataframe):
    sentences, labels = [], []  # Initialize lists for sentences and labels
    for _, row in dataframe.iterrows():  # Iterate through each document
        for sent in sent_tokenize(str(row['revised'])):  # Split each document into sentences
            if len(sent.strip()) > 3:  # Ignore sentences that are too short
                sentences.append(sent)  # Add the sentence to the list
                labels.append(row['target'])  # Assign the document-level label to the sentence
    return sentences, np.array(labels)  # Return sentences and their corresponding labels

train_sentences, y_train_ground = extract_sentences_from_df(train_df_balanced)  # Extract training sentences and labels
test_sentences, y_test_ground = extract_sentences_from_df(test_df)  # Extract testing sentences and labels

# =====================================================================
# STEP 4: ELECTRA - MAJORITY VOTING AND SIMULTANEOUS EMBEDDING EXTRACTION
# =====================================================================

model_checkpoint = "google/electra-small-discriminator"  # Specify the ELECTRA checkpoint
print(f"\n⏳ Loading {model_checkpoint}...")  # Display the model loading status

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)  # Load the pretrained ELECTRA tokenizer
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=2)  # Load ELECTRA with a token classification head
model.to(device)  # Move the model to the selected device
model.eval()  # Set the model to evaluation mode

def process_with_electra_and_voting(sentences):
    embeddings = []  # Store sentence embeddings
    voted_labels = []  # Store sentence-level majority voting predictions

    with torch.no_grad():  # Disable gradient calculation to reduce memory usage
        for sent in sentences:  # Process each sentence individually
            inputs = tokenizer(sent, return_tensors="pt", truncation=True, max_length=128).to(device)  # Tokenize and move inputs to the selected device

            # Perform a single forward pass through the ELECTRA backbone
            outputs_base = model.electra(**inputs)  # Extract contextual token representations
            logits = model.classifier(outputs_base.last_hidden_state)  # Generate classification scores for each token

            # 1. Token-level Majority Voting -> Sentence-level Prediction
            preds = torch.argmax(logits, dim=2)[0].cpu().numpy()  # Select the predicted class for each token
            valid_preds = [p for p in preds if p in [0, 1]]  # Keep only valid binary predictions

            if not valid_preds:
                v_label = 0  # Assign the default Human label if no valid predictions exist
            else:
                counts = {0: valid_preds.count(0), 1: valid_preds.count(1)}  # Count predictions for each class
                v_label = max(counts, key=counts.get)  # Select the class with the highest vote count

            # 2. Mean Pooling for Sentence Embedding
            emb = outputs_base.last_hidden_state.mean(dim=1).cpu().numpy()[0]  # Average token embeddings to obtain a sentence representation

            voted_labels.append(v_label)  # Store the majority voting prediction
            embeddings.append(emb)  # Store the sentence embedding

    return np.array(embeddings), np.array(voted_labels)  # Return embeddings and voting predictions as NumPy arrays

print("🗳️ Processing training set (Voting + Embedding)...")  # Display the training processing status
X_train, y_train_voted = process_with_electra_and_voting(train_sentences)  # Extract training embeddings and voting predictions

print("🗳️ Processing testing set (Voting + Embedding)...")  # Display the testing processing status
X_test, y_test_voted = process_with_electra_and_voting(test_sentences)  # Extract testing embeddings and voting predictions

# =====================================================================
# STEP 5: TRAIN SVM AND OPTIMIZE THE DECISION THRESHOLD
# =====================================================================

print("\n🤖 Training the SVM classifier (RBF/Linear kernel balancing)...")  # Display the SVM training status

# Use the RBF kernel to model nonlinear decision boundaries
svm_classifier = SVC(
    kernel='rbf',  # Use the Radial Basis Function kernel
    C=1.0,  # Set the regularization parameter
    probability=True,  # Enable probability estimation
    class_weight='balanced',  # Automatically adjust weights for imbalanced classes
    random_state=42  # Ensure reproducible results
)

svm_classifier.fit(X_train, y_train_ground)  # Train the SVM using sentence embeddings and ground-truth labels

# Get predicted probabilities for the Human class (class 0)
probs = svm_classifier.predict_proba(X_test)[:, 0]  # Extract the probability of each sentence belonging to the Human class

# Find the optimal threshold for the Human class to maximize its F1-score
precisions, recalls, thresholds = precision_recall_curve(y_test_ground == 0, probs)  # Calculate precision and recall at different thresholds
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)  # Calculate F1-scores while avoiding division by zero
best_idx = np.argmax(f1_scores)  # Find the index of the highest F1-score
best_threshold = thresholds[best_idx]  # Retrieve the corresponding decision threshold

print(f"🎯 Optimal decision threshold for the Human class: {best_threshold:.4f}")  # Display the selected threshold

# Predict class 0 if the Human probability >= best_threshold; otherwise, predict class 1
y_pred_tuned = np.where(probs >= best_threshold, 0, 1)  # Apply the optimized decision threshold

# =====================================================================
# STEP 6: EVALUATE THE IMPROVED MODEL
# =====================================================================

acc = accuracy_score(y_test_ground, y_pred_tuned)  # Calculate the overall classification accuracy

print("\n" + "="*55)  # Print a separator
print("📈 EVALUATION RESULTS AFTER IMPROVEMENT (UNDERSAMPLING + TUNING):")  # Display the evaluation heading
print("="*55)  # Print a separator

print(f"👉 Accuracy: {acc * 100:.2f}%\n")  # Display accuracy as a percentage
print("Detailed Classification Report:")  # Display the classification report heading

print(classification_report(
    y_test_ground,  # Provide the ground-truth labels
    y_pred_tuned,  # Provide the predicted labels
    target_names=['Human (HWT)', 'AI/Mixed (MGT)'],  # Specify the class names
    zero_division=0  # Return zero when a metric is undefined
))


🚀 Using device: cuda
✅ MixSet dataset loaded: 3000 rows.

📊 Training set distribution AFTER undersampling:
target
1    600
0    400
Name: count, dtype: int64

📊 Test set distribution (original distribution):
target
1    500
0    100
Name: count, dtype: int64

⏳ Loading google/electra-small-discriminator...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 54.2MB            

... (3 outputs left out)

🗳️ Processing training set (Voting + Embedding)...


model.safetensors: reconstructing file:   0%|          |  0.00B / 54.2MB            

model.safetensors: downloading bytes:           |  0.00B            

🗳️ Processing testing set (Voting + Embedding)...

🤖 Training the SVM classifier (RBF/Linear kernel balancing)...
🎯 Optimal decision threshold for the Human class: 0.7959

📈 EVALUATION RESULTS AFTER IMPROVEMENT (UNDERSAMPLING + TUNING):
👉 Accuracy: 93.70%

Detailed Classification Report:
                precision    recall  f1-score   support

   Human (HWT)       0.90      0.85      0.87      1099
AI/Mixed (MGT)       0.95      0.97      0.96      3280

      accuracy                           0.94      4379
     macro avg       0.92      0.91      0.91      4379
  weighted avg       0.94      0.94      0.94      4379

